# Chapter 2 — Benchmark scores as statistical estimates

Companion notebook for *Evaluating Large Language Models in High-Stakes Domains* (Costa, 2026). All data in this notebook are simulated; the generating process is in the code and described in the text.

In [ ]:
"""Figures and worked-example numbers for 'Evaluating Large Language Models in High-Stakes Domains'.
All data are analytic or simulated; generating processes are stated in the text.
Run:  python figures.py   (writes PNGs and results.json into ./figures)
"""
import os, json
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import stats
from scipy.optimize import minimize, minimize_scalar

OUT = os.path.join(os.getcwd(), "figures_out")
os.makedirs(OUT, exist_ok=True)

# palette (categorical, fixed order) and ink
C1, C2, C3 = "#2a78d6", "#eb6834", "#1baf7a"
INK, INK2, GRID = "#0b0b0b", "#52514e", "#d9d8d4"
plt.rcParams.update({
    "font.family": "DejaVu Sans", "font.size": 9,
    "axes.edgecolor": INK2, "axes.labelcolor": INK, "xtick.color": INK2,
    "ytick.color": INK2, "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.6,
    "legend.frameon": False, "figure.dpi": 200, "savefig.dpi": 200,
    "savefig.facecolor": "white",
})
results = {}
se = lambda x: x.std(ddof=1)/np.sqrt(len(x))
def wilson(x, n, z=1.96):
    p = x/n; c = (p + z*z/(2*n))/(1 + z*z/n)
    h = z*np.sqrt(p*(1-p)/n + z*z/(4*n*n))/(1 + z*z/n)
    return c-h, c+h
def clopper_upper(x, n, conf=0.95):
    return 1.0 if x == n else stats.beta.ppf(conf, x+1, n-x)


## Figure 1

In [ ]:
def wilson_halfwidth(p, n, z=1.96):
    return z*np.sqrt(p*(1-p)/n + z**2/(4*n**2)) / (1 + z**2/n)

n = np.arange(20, 1501)
fig, ax = plt.subplots(1, 2, figsize=(7.2, 2.9))
for p, c in zip([0.70, 0.85, 0.95], [C1, C2, C3]):
    ax[0].plot(n, 100*wilson_halfwidth(p, n), color=c, lw=2, label=f"accuracy {p:.2f}")
ax[0].set_xlabel("Number of items n"); ax[0].set_ylabel("95% CI half-width (points)")
ax[0].set_title("(a) Precision of a single accuracy", loc="left", fontsize=9, color=INK)
ax[0].set_xlim(0, 1550); ax[0].legend(loc="upper right")
zs = 1.96 + 0.8416
for disc, c in zip([0.10, 0.20, 0.30], [C1, C2, C3]):
    d = np.sqrt(zs**2*disc/(n + zs**2))
    ax[1].plot(n, 100*d, color=c, lw=2, label=f"discordance {disc:.2f}")
ax[1].set_xlabel("Number of items n"); ax[1].set_ylabel("Minimum detectable difference (points)")
ax[1].set_title("(b) Paired comparison, 80% power", loc="left", fontsize=9, color=INK)
ax[1].set_xlim(0, 1550); ax[1].legend(loc="upper right")
fig.tight_layout(); fig.savefig(os.path.join(OUT, "fig1_ci_width.png")); plt.close(fig)

## Worked example, Ch. 2

In [ ]:
rng = np.random.default_rng(7)
n_, k = 120, 5
item_effect = rng.normal(0, 1.6, n_)
logit_A = 2.0 + item_effect + rng.normal(0, .3, n_)
logit_B = 1.4 + item_effect + rng.normal(0, .3, n_)
pA, pB = 1/(1+np.exp(-logit_A)), 1/(1+np.exp(-logit_B))
sA = rng.binomial(1, pA[:, None], (n_, k)); sB = rng.binomial(1, pB[:, None], (n_, k))
mA, mB = sA.mean(1), sB.mean(1)
d = mA - mB
B = 5000; idx = rng.integers(0, n_, (B, n_)); boot = (mA[idx]-mB[idx]).mean(1)
lo, hi = np.percentile(boot, [2.5, 97.5])
results["ch2"] = {
    "A_ACC": f"{mA.mean():.3f} [{mA.mean()-1.96*se(mA):.3f}, {mA.mean()+1.96*se(mA):.3f}]",
    "B_ACC": f"{mB.mean():.3f} [{mB.mean()-1.96*se(mB):.3f}, {mB.mean()+1.96*se(mB):.3f}]",
    "A_NAIVE": f"{se(sA.ravel()):.4f}", "A_CLUST": f"{se(mA):.4f}",
    "PAIRED": f"{d.mean():.3f} [{d.mean()-1.96*se(d):.3f}, {d.mean()+1.96*se(d):.3f}]",
    "UNPAIRED_SE": f"{np.sqrt(se(mA)**2+se(mB)**2):.4f}", "PAIRED_SE": f"{se(d):.4f}",
    "BOOT": f"[{lo:.3f}, {hi:.3f}]",
}

In [ ]:
import json
print(json.dumps(results, indent=2, ensure_ascii=False))